# Clustering Challenge — final solution

Task: cluster **24 hidden datasets** (ds00–ds23). Metric: **Mean ARI** against the hidden true labels (each dataset weighted equally; pass line 0.60). The hidden test is a stratified ~20% sample of each dataset, so the number of true clusters in the test equals the number in the full population — this drives all per-dataset choices below.


## Данные участника
ФИО: Борцов Михаил Алексеевич
Email: mialbortsov@edu.hse.ru
ВУЗ: ВШЭ


## Solution

`solution.py` implements a **per-dataset recipe**: for every dataset_id the preprocessing, the number of clusters k and the clustering algorithm are chosen individually (the task statement explicitly says one universal pipeline is not expected to be optimal). The same code path is used at inference time; nothing is fitted on hidden data except the final clustering itself.

**1. Cleaning.** Per dataset, trailing all-NaN columns are dropped (the README says unused trailing x-columns are empty); any remaining NaN cells are filled with the column mean. This makes every downstream step NaN-free.

**2. Per-dataset configuration (k and model).** Chosen by comparing candidate settings on test-size resamples of the public train (label stability across slices, silhouette in the reduced space) and validated on the leaderboard:

| ds | k | model | ds | k | model |
|---|---|---|---|---|---|
| ds00 | 3 | KMeans | ds12 | 2 | GMM (full cov.) |
| ds01 | 3 | GMM (full cov.) | ds13 | 3 | GMM (tied cov.) |
| ds02 | 2 | Spectral (knn) | ds14 | 2 | GMM (tied cov.) |
| ds03 | 2 | GMM (full cov.) | ds15 | 4 | GMM (full cov.) |
| ds04 | 3 | GMM (full cov.) | ds16 | 2 | ring/blob boundary |
| ds05 | 3 | GMM (full cov.) | ds17 | 2 | KMeans |
| ds06 | 3 | KMeans | ds18 | 2 | GMM (full cov.) |
| ds07 | 3 | Spectral (knn) | ds19 | 3 | Spectral (knn) |
| ds08 | 2 | GMM (full cov.) | ds20 | 4 | KMeans |
| ds09 | 3 | GMM (tied cov.) | ds21 | 3 | Spectral (knn) |
| ds10 | 3 | GMM (full cov.) | ds22 | 2 | GMM (full cov.) |
| ds11 | 4 | GMM (full cov.) | ds23 | 2 | ring/blob boundary |

GMM = GaussianMixture with reg_covar=1e-3, n_init=5; KMeans n_init=50; SpectralClustering with nearest-neighbours affinity (n_neighbors=20). All fits are seeded (random_state=42), so the solution is fully deterministic.

**3. Transductive pooling for 7 hard datasets.** For ds02, ds05, ds08, ds13, ds14, ds15, ds18 the test sample alone (~244 rows) is too small to fit a stable partition. For these datasets the public train (80% split) is preprocessed once — per-dataset normalizer + PCA — and embedded in `solution.py` as float16 hex (`POOL_EMBED`). At inference time the hidden test rows are transformed with the *same stored* normalizer/PCA parameters, concatenated with the embedded train cloud, clustered together, and only the test-row labels are returned. The other 17 datasets are clustered on the test sample alone (their structure is already resolved at n≈244, and pooling did not improve their slice-stability).

**4. Local preprocessing.** For non-pooled datasets: MinMax scaling for ds00, ds01, ds05, ds06, ds12, ds17, ds22 and Standard scaling otherwise; then PCA down to the dimension covering ~95% of variance (stored per dataset in `EVR_95`) where it reduces the dimensionality.

**5. Ring + central blob datasets (ds16, ds23).** These have an elliptical ring around a dense central blob; generic k=2 clusterers split the ring into arcs. Instead, an ellipse boundary is fitted in PCA-2 space: the outer 45% of points define a 36-bin angular radius profile (median distance per bin, 5-tap smoothing), and each point is labelled inside/outside by comparing its distance to `frac * R(angle)` with frac = 0.85 (ds16) / 0.72 (ds23). If the inside fraction falls outside [0.15, 0.85] the boundary is rejected and a plain KMeans-2 on radial distance is used as fallback.


## Выводы

- Универсальный пайплайн не работает: для каждого из 24 наборов подобраны свои k, модель и предобработка (таблица выше), что подтверждено экспериментами на ресемплах размера тестовой выборки и на лидерборде.
- Для 7 трудных наборов (ds02, ds05, ds08, ds13, ds14, ds15, ds18) применено трансдуктивное пулирование: публичный train преобразуан теми же параметрами (нормализатор + PCA), что и скрытый test, и кластеризация идёт на объединённом облаке — это стабилизировало разбиение на малых тестовых выборках.
- Для ds16/ds23 (кольцо + центральное пятно) вместо k=2 кластера использована эллиптическая граница «внутри/снаружи» в PCA-2 пространстве, что устраняет разрезание кольца на дуги.
- Решение детерминировано (все случайные алгоритмы зафиксированы random_state=42), работает только на разрешённых библиотеках (numpy, sklearn) и укладывается в лимит времени (~1 с на все 24 набора).
